<a href="https://colab.research.google.com/github/sahanasb/Intelligent-E-Commerce-Search-with-Retrieval-Augmented-Generation/blob/main/BM25%2BSBERT%2BHybridSearch%2BRedis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas rank_bm25 sentence-transformers scikit-learn

In [ ]:
!pip install langchain langchain-community langchain-groq sentence-transformers chromadb --break-system-packages

In [ ]:
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import torch
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np
import shutil
import os

# loading data
df = pd.read_csv('/content/clean_data.csv')
df.head()

In [ ]:
# Vector embedding
def build_doc(row, index):
    return Document(
        page_content=str(row['search_text']),
        metadata={
            # add index for Hybrid Search
            "idx": index,
            "name": str(row['name']),
            "category": str(row['main_category']),
            "price_usd": float(row['actual_price_usd']),
            "ratings": float(row['ratings'])
        }
    )
docs = [build_doc(row, index) for index, row in df.iterrows()]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"})

In [ ]:
if os.path.exists("./product_db"):
    shutil.rmtree("./product_db")
    print("Deleted old product_db")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./product_db"
)

In [ ]:
results = vectorstore.similarity_search("i am planning to go to beach")

for i, doc in enumerate(results[:3]):
    print(f"\n--- Document {i+1} ---")
    print("Name:", doc.metadata.get("name"))
    print("Category:", doc.metadata.get("category"))
    print("Price:", doc.metadata.get("price_usd"))
    print("Rating:", doc.metadata.get("ratings"))
    print("Content:", doc.page_content)

#BM25

In [ ]:
# Tokenization
tokenized_corpus = [doc.page_content.lower().split() for doc in docs]

# Initialization
bm25 = BM25Okapi(tokenized_corpus)

# BM25 query
def bm25_search(query, top_k=5):
  # Preprocess the user's query
  tokenized_query = query.lower().split()

  # Calculate scores for all documents in the corpus
  bm25_scores = bm25.get_scores(tokenized_query)

  # Get indices of top_k results
  top_indices = np.argsort(bm25_scores)[::-1][:top_k]
  print(f"Query: {query}")

  for i in top_indices:
      print(f"Score: {bm25_scores[i]:.2f} | Name: {df.iloc[i]['name']}")
  return top_indices, bm25_scores
# Test BM25
indices, scores = bm25_search("i am planning to go to beach")

#SBERT

In [ ]:
#SBERT
def sbert_search(query, top_k=5):
  # convert a single string of text into a vector
  query_embedding = embeddings.embed_query(query)

  # finds the closest matches using metrics like cosine similarity
  results = vectorstore.similarity_search(query, k=top_k)
  return results

# Test SBERT
query = "i am planning to go to beach"
sbert_results = sbert_search(query)
for doc in sbert_results:
    print(f"- {doc.metadata['name']}")

#Hybrid Search

In [ ]:
def hybrid_search(query, top_k=5):
  # get BM25 rank
  tokenized_query = query.lower().split()
  bm25_scores = bm25.get_scores(tokenized_query)
  bm25_top_indices = np.argsort(bm25_scores)[::-1]
  bm25_mapping = {idx: rank for rank, idx in enumerate(bm25_top_indices)}

  # get SBERT rank
  sbert_results = vectorstore.similarity_search(query, k=100)

  sbert_mapping = {}
  for rank, doc in enumerate(sbert_results):
    idx = doc.metadata.get('idx')
    sbert_mapping[idx] = rank

  # Reciprocal Rank Fusion (RRF) to combine two ranked results
  rrf_scores = {}
  k = 60
  all_candidate_indices = set(list(bm25_top_indices[:100]) + list(sbert_mapping.keys()))

  for idx in all_candidate_indices:
    rank_bm = bm25_mapping.get(idx, 1000)
    rank_sbert = sbert_mapping.get(idx, 1000)

    score = (1 / (k + rank_bm)) + (1 / (k + rank_sbert))
    rrf_scores[idx] = score
  # sort score
  final_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
  return [docs[i] for i in final_indices]

# Test hybrid search
querty = "i am planning to go to beach"
hybrid_results = hybrid_search(query)
for doc in hybrid_results:
    print(f"- {doc.metadata['name']}")

# Redis

In [ ]:
pip install redis

In [ ]:
!apt-get install redis-server > /dev/null
!redis-server --daemonize yes

In [ ]:
import redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
try:
  if r.ping():
    print("connect redis successfully")
except Exception as e:
    print("not connect redis")

In [ ]:
import json
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
def final_search_pipeline(query, top_k=5):
  start_time = time.time()

  # check redis cache
  cache_key = f"search:{query.lower().strip()}"
  cached_res = r.get(cache_key)

  if cached_res:
    latency = (time.time() - start_time) * 1000
    print(f"CATCH HIT running: {latency:.2f}ms")
    return json.loads(cached_res)

  # query not in cache
  initial_candidates = hybrid_search(query, top_k=50)
  final_results = initial_candidates[:top_k]

  # TODO rerank_result then switch initial_candidates[:top_k] to rerank_results(query, initial_candidates, top_n=top_k)

  #final_results = rerank_results(query, initial_candidates, top_n=top_k)
  output = []
  for doc in final_results:
    output.append({
        "name": doc.metadata.get("name"),
        "price": doc.metadata.get("price_usd"),
        "category": doc.metadata.get("category"),
        "idx": doc.metadata.get("idx")
    })
  r.setex(cache_key, 3600, json.dumps(output))
  return output
results = final_search_pipeline("i am planning to go to beach")